In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send, interrupt, Command
from typing import TypedDict
import subprocess
from openai import OpenAI
import textwrap
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator
import base64
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

llm = init_chat_model("openai:gpt-4o-mini")

class State(TypedDict):
    audio_file: str
    transcription: str

def record_audio(state: State):
    current_timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    output_file = "voices/recorded_audio.mp4"
    command = [
        "ffmpeg",
        "-f",
        "avfoundation",
        "-i",
        ":0",
        "-t",
        "00:01:00",
        "-y",
        output_file,
    ]
    subprocess.run(command)
    return {
        "audio_file": output_file,
    }

def transcribe_audio(state: State):
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            language="en",
            prompt="Transcribe the audio for an English speaking test (TOEIC Speaking/OPIc-style picture description). The transcription should be clear and accurate, capturing all spoken words, including any hesitations, fillers (like 'um', 'uh'), and natural speech patterns. Ensure the transcription is suitable for evaluating English speaking skills, with proper punctuation and formatting to reflect the speaker's delivery."
        )
        return {
            "transcription": transcription,
        }

graph_builder = StateGraph(State)
graph_builder.add_node("record_audio", record_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_edge(START, "record_audio")
graph_builder.add_edge("record_audio", "transcribe_audio")
graph_builder.add_edge("transcribe_audio", END)
graph = graph_builder.compile()
graph.invoke()